In [2]:
import pandas as pd
import numpy as np
from datetime import datetime

## Read training and test datasets

In [4]:
# helper function for reading datatset
def read_data(file_path):
    df = pd.read_csv(file_path)
    df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d') # convert it to datatime

    return df

In [5]:
df_train = read_data('data/kospi_train.csv')
df_test = read_data('data/kospi_test.csv')

len(df_train), len(df_test)

(986, 244)

In [6]:
# the training dataset has daily KOSPI index from 2019 to 2022
df_train

,Date,Open,Low,High,Close,Volume
0,2019-01-02,2050.550049,2004.270020,2053.449951,2010.000000,326400
1,2019-01-03,2011.810059,1991.650024,2014.719971,1993.699951,428000
2,2019-01-04,1992.400024,1984.530029,2011.560059,2010.250000,409000
3,2019-01-07,2034.239990,2030.900024,2048.060059,2037.099976,440200
4,2019-01-08,2038.680054,2023.589966,2042.699951,2025.270020,397800
...,...,...,...,...,...,...
981,2022-12-23,2325.860107,2311.899902,2333.080078,2313.689941,367000
982,2022-12-26,2312.540039,2304.199951,2321.919922,2317.139893,427600
983,2022-12-27,2327.520020,2321.479980,2335.989990,2332.790039,448300
984,2022-12-28,2296.449951,2276.899902,2296.449951,2280.449951,405700


In [7]:
# the test dataset has daily KOSPI index in 2023
df_test

,Date,Open,Low,High,Close,Volume
0,2023-01-02,2249.949951,2222.370117,2259.879883,2225.669922,346100
1,2023-01-03,2230.979980,2180.669922,2230.979980,2218.679932,410000
2,2023-01-04,2205.979980,2198.820068,2260.060059,2255.979980,412700
3,2023-01-05,2268.199951,2252.969971,2281.389893,2264.649902,430800
4,2023-01-06,2253.399902,2253.270020,2300.620117,2289.969971,398300
...,...,...,...,...,...,...
239,2023-12-21,2598.370117,2587.159912,2610.810059,2600.020020,578300
240,2023-12-22,2617.719971,2599.510010,2621.370117,2599.510010,466000
241,2023-12-26,2609.439941,2594.649902,2612.139893,2602.590088,439500
242,2023-12-27,2599.350098,2590.080078,2613.500000,2613.500000,349700


## Part 1. Train regression models to predict the next day's `close` using `Open`, `Low`, `High`, `Close`, `Volume` of previous days as predictors using *only* df_train. Cross-validate to select the best model. Evaluate the accuracy of your model using `df_test`.


In [9]:
# GOOD LUCK
#import library
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.api as sm
from itertools import combinations
from sklearn.model_selection import TimeSeriesSplit
import matplotlib.pyplot as plt

In [10]:
#setting data structure
features = ['Open', 'Low','High','Close','Volume']
df_train['Target_Close'] = df_train['Close'].shift(-1)
df_train.dropna(inplace=True)
X = df_train[features].values
y = df_train['Target_Close'].values
results = []
summary_data = []
rmse_list, mae_list, r2_list = [], [], []

In [11]:
def cv_rmse_r2(X, y, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rmse_scores = []
    r2_scores = []
    for train_idx, val_idx in tscv.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = LinearRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        rmse_scores.append(mean_squared_error(y_val, y_pred, squared=False))
        r2_scores.append(r2_score(y_val, y_pred))

    return (
        np.mean(rmse_scores), np.std(rmse_scores),
        np.mean(r2_scores), np.std(r2_scores)
    )

### Backward Selection

In [13]:
#Using all features at start
rmse_mean, rmse_std, r2_mean, r2_std = cv_rmse_r2(X, y)
print(f"RMSE: {rmse_mean:.4f} ± {rmse_std:.4f}")
print(f"R²: {r2_mean:.4f} ± {r2_std:.4f}")

RMSE: 32.4581 ± 3.1036
R²: 0.9404 ± 0.0440


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use 

In [14]:
result_table = pd.DataFrame([{
    'Step': 'Initial (All)',
    'Features Used': ', '.join(features),
    'CV RMSE': f"{rmse_mean:.4f} ± {rmse_std:.4f}",
    'CV R²': f"{r2_mean:.4f} ± {r2_std:.4f}"
}])

In [15]:
#we can drop feature one by one if rmse doesn't increase
selected_features = features.copy()
base_rmse, base_std, base_r2, base_r2_std = cv_rmse_r2(X, y)
result_table = pd.DataFrame([{
    'Step': 'Initial (All)',
    'Features Used': ', '.join(selected_features),
    'CV RMSE': f"{base_rmse:.4f} ± {base_std:.4f}",
    'CV R²': f"{base_r2:.4f} ± {base_r2_std:.4f}"
}])

# Try dropping each feature one by one
for feature in features:
    # Try removing current feature
    temp_features = [f for f in selected_features if f != feature]
    X_temp = df_train[temp_features].values
    temp_rmse, temp_std, temp_r2, temp_r2_std = cv_rmse_r2(X_temp, y)

    print(f"\nTry dropping '{feature}'")
    print(f"RMSE: {temp_rmse:.4f} ± {temp_std:.4f}")
    print(f"R²  : {temp_r2:.4f} ± {temp_r2_std:.4f}")
    
    # If RMSE improves or stays the same, drop the feature
    if temp_rmse <= base_rmse:
        print(f"→ Dropping '{feature}' improved or maintained RMSE. Removing.")
        selected_features = temp_features
        base_rmse, base_std = temp_rmse, temp_std
        base_r2, base_r2_std = temp_r2, temp_r2_std
        step_name = f"Dropped {feature}"
    else:
        print(f"→ Dropping '{feature}' worsened RMSE. Keeping it.")
        step_name = f"Tried dropping {feature} (kept)"

    # Append result to the table
    result_table.loc[len(result_table)] = {
        'Step': step_name,
        'Features Used': ', '.join(temp_features if temp_rmse <= base_rmse else selected_features),
        'CV RMSE': f"{temp_rmse:.4f} ± {temp_std:.4f}",
        'CV R²': f"{temp_r2:.4f} ± {temp_r2_std:.4f}"
    }


Try dropping 'Open'
RMSE: 32.4717 ± 3.0655
R²  : 0.9402 ± 0.0441
→ Dropping 'Open' worsened RMSE. Keeping it.

Try dropping 'Low'
RMSE: 32.3469 ± 2.9909
R²  : 0.9405 ± 0.0442
→ Dropping 'Low' improved or maintained RMSE. Removing.

Try dropping 'High'
RMSE: 31.9307 ± 2.2926
R²  : 0.9410 ± 0.0446
→ Dropping 'High' improved or maintained RMSE. Removing.

Try dropping 'Close'
RMSE: 40.8455 ± 3.9388
R²  : 0.9029 ± 0.0769
→ Dropping 'Close' worsened RMSE. Keeping it.

Try dropping 'Volume'
RMSE: 31.0933 ± 1.8954
R²  : 0.9424 ± 0.0444
→ Dropping 'Volume' improved or maintained RMSE. Removing.


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use 

In [16]:
result_table

,Step,Features Used,CV RMSE,CV R²
0,Initial (All),"Open, Low, High, Close, Volume",32.4581 ± 3.1036,0.9404 ± 0.0440
1,Tried dropping Open (kept),"Open, Low, High, Close, Volume",32.4717 ± 3.0655,0.9402 ± 0.0441
2,Dropped Low,"Open, High, Close, Volume",32.3469 ± 2.9909,0.9405 ± 0.0442
3,Dropped High,"Open, Close, Volume",31.9307 ± 2.2926,0.9410 ± 0.0446
4,Tried dropping Close (kept),"Open, Close, Volume",40.8455 ± 3.9388,0.9029 ± 0.0769
5,Dropped Volume,"Open, Close",31.0933 ± 1.8954,0.9424 ± 0.0444


### Now we can add mixed selection!

In [18]:
candidates = ['High', 'Low', 'Volume']
y = df_train['Target_Close'].values

# Base model with current selected features
X_base = df_train[selected_features].values
base_rmse, base_std, base_r2, base_r2_std = cv_rmse_r2(X_base, y)

print(f"[Base Model] Features: {selected_features}")
print(f"CV RMSE: {base_rmse:.4f} ± {base_std:.4f}")
print(f"CV R²  : {base_r2:.4f} ± {base_r2_std:.4f}")

# Log the base model in result table
result_table.loc[len(result_table)] = {
    'Step': 'Base Model (Before Adding)',
    'Features Used': ', '.join(selected_features),
    'CV RMSE': f"{base_rmse:.4f} ± {base_std:.4f}",
    'CV R²': f"{base_r2:.4f} ± {base_r2_std:.4f}"
}

# Try appending candidate features one by one
for feature in candidates:
    trial_features = selected_features + [feature]
    X_trial = df_train[trial_features].values

    # Evaluate cross-validation performance
    trial_rmse, trial_std, trial_r2, trial_r2_std = cv_rmse_r2(X_trial, y)

    # Evaluate statistical significance using p-value
    X_with_const = np.hstack([np.ones((X_trial.shape[0], 1)), X_trial])
    sm_model = sm.OLS(y, X_with_const).fit()
    p_value = sm_model.pvalues[-1]  # p-value of newly added feature

    print(f"\n[Try Adding '{feature}']")
    print(f"CV RMSE: {trial_rmse:.4f} ± {trial_std:.4f}, p-value: {p_value:.4f}")

    # Add the feature only if RMSE improves and p-value is significant
    if trial_rmse < base_rmse and p_value < 0.05:
        print("→ Append!")
        selected_features.append(feature)
        base_rmse, base_std = trial_rmse, trial_std
        base_r2, base_r2_std = trial_r2, trial_r2_std
        step_name = f"Added {feature}"
    else:
        print("→ Don't append.")
        step_name = f"Tried adding {feature} (not used)"

    # Record the result of this step
    result_table.loc[len(result_table)] = {
        'Step': step_name,
        'Features Used': ', '.join(trial_features if trial_rmse < base_rmse and p_value < 0.05 else selected_features),
        'CV RMSE': f"{trial_rmse:.4f} ± {trial_std:.4f}",
        'CV R²': f"{trial_r2:.4f} ± {trial_r2_std:.4f}"
    }

[Base Model] Features: ['Open', 'Close']
CV RMSE: 31.0933 ± 1.8954
CV R²  : 0.9424 ± 0.0444

[Try Adding 'High']
CV RMSE: 31.6445 ± 2.1143, p-value: 0.0016
→ Don't append.

[Try Adding 'Low']
CV RMSE: 31.5226 ± 1.8641, p-value: 0.2126
→ Don't append.

[Try Adding 'Volume']
CV RMSE: 31.9307 ± 2.2926, p-value: 0.0463
→ Don't append.


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use 

In [19]:
selected_features

['Open', 'Close']

In [20]:
# Assume df_test exists and is preprocessed in the same way as df_train
df_test['Target_Close'] = df_train['Close'].shift(-1)
df_test.dropna(inplace=True)
X_test = df_test[selected_features].values
y_test = df_test['Target_Close'].values

# Train on full train set
X_train_final = df_train[selected_features].values
y_train_final = df_train['Target_Close'].values

model = LinearRegression()
model.fit(X_train_final, y_train_final)

# Predict on test set
y_pred = model.predict(X_test)

# Evaluate
test_rmse = mean_squared_error(y_test, y_pred, squared=False)
test_mae = mean_absolute_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)

result_table.loc[len(result_table)] = {
    'Step': 'Final Test Evaluation',
    'Features Used': ', '.join(selected_features),
    'CV RMSE': f"N/A (Test only)",
    'CV R²': f"{test_r2:.4f}"
}

print(f"Test RMSE : {test_rmse:.4f}")
print(f"Test MAE  : {test_mae:.4f}")
print(f"Test R²   : {test_r2:.4f}")

Test RMSE : 413.2676
Test MAE  : 393.1627
Test R²   : -26.4498


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [21]:
result_table

,Step,Features Used,CV RMSE,CV R²
0,Initial (All),"Open, Low, High, Close, Volume",32.4581 ± 3.1036,0.9404 ± 0.0440
1,Tried dropping Open (kept),"Open, Low, High, Close, Volume",32.4717 ± 3.0655,0.9402 ± 0.0441
2,Dropped Low,"Open, High, Close, Volume",32.3469 ± 2.9909,0.9405 ± 0.0442
3,Dropped High,"Open, Close, Volume",31.9307 ± 2.2926,0.9410 ± 0.0446
4,Tried dropping Close (kept),"Open, Close, Volume",40.8455 ± 3.9388,0.9029 ± 0.0769
5,Dropped Volume,"Open, Close",31.0933 ± 1.8954,0.9424 ± 0.0444
6,Base Model (Before Adding),"Open, Close",31.0933 ± 1.8954,0.9424 ± 0.0444
7,Tried adding High (not used),"Open, Close",31.6445 ± 2.1143,0.9418 ± 0.0439
8,Tried adding Low (not used),"Open, Close",31.5226 ± 1.8641,0.9419 ± 0.0439
9,Tried adding Volume (not used),"Open, Close",31.9307 ± 2.2926,0.9410 ± 0.0446


### Poor prediction

### What if...?<br> Other 3 values are can't append because of p-value <br> however, what if we add not only one by one but two or three by two or three...?

### We have only 5 features in Part 1<br>5 features's possible combinations count is 30 (5C1 + 5C2 + 5C3 + 5C4 + 5C5)<br>in this case we can use BruteForce Selection.<br>we can check them all!

In [25]:
for i in range(1, 6):
    features_combination = list(combinations(features, i))
    
    for feature_set in features_combination:
        X = df_train[list(feature_set)].values
        y = df_train['Target_Close'].values

        #cv
        rmse_mean, rmse_std, r2_mean, r2_std = cv_rmse_r2(X, y)

        
        results.append({
            'Feature': feature_set,
            'RMSE_mean': rmse_mean,
            'RMSE_std': rmse_std,
            'R2_mean': r2_mean,
            'R2_std': r2_std
        })

#make dataframe
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='RMSE_mean').reset_index(drop=True)


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use 

In [26]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='RMSE_mean').reset_index(drop=True)
results_df

,Feature,RMSE_mean,RMSE_std,R2_mean,R2_std
0,"(Close,)",31.066538,1.910614,0.942431,0.044433
1,"(Open, Close)",31.093316,1.895369,0.942406,0.044405
2,"(Low, Close)",31.187180,1.829083,0.942312,0.044208
3,"(High, Close)",31.298813,1.880872,0.942239,0.044267
4,"(Open, Low, Close)",31.522632,1.864147,0.941924,0.043924
5,"(Open, High, Close)",31.644514,2.114299,0.941845,0.043944
6,"(Low, High, Close)",31.774419,2.157264,0.941680,0.043661
7,"(Open, Low, High, Close)",31.782993,2.207254,0.941707,0.043635
8,"(Close, Volume)",31.898646,2.262027,0.941036,0.044646
9,"(Open, Close, Volume)",31.930717,2.292561,0.941012,0.044578


## Part 2. Extend the regression model by adding some extra features of your choice. You can use any statistics publicly available. 

In [28]:
# GOOD LUCK
#First we need to check our data shape
len(df_train), len(df_test)

(985, 244)

In [29]:
df_train.shape

(985, 7)

In [30]:
df_test.shape

(244, 7)

In [31]:
#add new RSI factor
def compute_rsi(series, period=14):
     # Calculate daily price changes
    delta = series.diff()

    # Separate gains (positive changes) and losses (negative changes)
    gain = delta.clip(lower=0)          # gains only
    loss = -delta.clip(upper=0)         # losses as positive values

    # Compute rolling average gains and losses over the specified period
    avg_gain = gain.rolling(window=period, min_periods=period).mean()
    avg_loss = loss.rolling(window=period, min_periods=period).mean()

    # Calculate Relative Strength (RS)
    rs = avg_gain / avg_loss

    # Calculate RSI using the standard formula
    rsi = 100 - (100 / (1 + rs))

    return rsi

In [32]:
def compute_momentum(series, period=10):
    # Subtract the closing price n days ago from today's closing price
    momentum = series - series.shift(period)
    return momentum

In [33]:
# Add RSI and Momentum to training data
df_train['RSI_14'] = compute_rsi(df_train['Close'], period=14)
df_train['Momentum_10'] = compute_momentum(df_train['Close'], period=10)

# Add RSI and Momentum to test data
df_test['RSI_14'] = compute_rsi(df_test['Close'], period=14)
df_test['Momentum_10'] = compute_momentum(df_test['Close'], period=10)

In [34]:
#Drop rows with NaN values
df_train.dropna(inplace=True)
df_test.dropna(inplace=True)

In [35]:
# All features (original + engineered)
all_features = ['Open', 'High', 'Low', 'Close', 'Volume', 'RSI_14', 'Momentum_10']
X = df_train[all_features].values
y = df_train['Target_Close'].values

# Cross-validation
full_rmse, full_std, full_r2, full_r2_std = cv_rmse_r2(X, y)
engineered_result_table = pd.DataFrame([{
    'Step': 'All features (original + engineered)',
    'Features Used': ', '.join(all_features),
    'CV RMSE': f"{full_rmse:.4f} ± {full_std:.4f}",
    'CV R²': f"{full_r2:.4f} ± {full_r2_std:.4f}"
}])
print("\n=== Cross-Validation Result with All Features ===")
print(f"Features used: {all_features}")
print(f"CV RMSE: {full_rmse:.4f} ± {full_std:.4f}")
print(f"CV R²  : {full_r2:.4f} ± {full_r2_std:.4f}")

/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(



=== Cross-Validation Result with All Features ===
Features used: ['Open', 'High', 'Low', 'Close', 'Volume', 'RSI_14', 'Momentum_10']
CV RMSE: 33.7506 ± 5.4155
CV R²  : 0.9375 ± 0.0458


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use 

In [36]:
# All possible initial features
selected_features = ['Open', 'High', 'Low', 'Close', 'Volume', 'RSI_14', 'Momentum_10']
y = df_train['Target_Close'].values

# Initial base performance
X = df_train[selected_features].values
base_rmse, base_std, base_r2, base_r2_std = cv_rmse_r2(X, y)

print(f"[Initial] Features: {selected_features}")
print(f"CV RMSE: {base_rmse:.4f} ± {base_std:.4f}")
print(f"CV R²  : {base_r2:.4f} ± {base_r2_std:.4f}")

# Begin feature dropping
for feature in selected_features.copy():
    temp_features = [f for f in selected_features if f != feature]
    X_temp = df_train[temp_features].values
    temp_rmse, temp_std, temp_r2, temp_r2_std = cv_rmse_r2(X_temp, y)

    print(f"\nTry dropping '{feature}'")
    print(f"CV RMSE: {temp_rmse:.4f} ± {temp_std:.4f}")

    if temp_rmse <= base_rmse:
        print(f"→ Dropping '{feature}' improved or maintained RMSE. Removing.")
        selected_features = temp_features
        base_rmse, base_r2 = temp_rmse, temp_r2
    else:
        print(f"→ Dropping '{feature}' worsened RMSE. Keeping.")


# Append final result after feature dropping
engineered_result_table.loc[len(engineered_result_table)] = {
    'Step': 'After Feature Dropping',
    'Features Used': ', '.join(selected_features),
    'CV RMSE': f"{base_rmse:.4f} ± {base_std:.4f}",
    'CV R²': f"{base_r2:.4f} ± {base_r2_std:.4f}"
}
# Final result
print("\n=== Final Selected Features ===")
print("Features:", selected_features)
print(f"CV RMSE: {base_rmse:.4f} ± {base_std:.4f}")
print(f"CV R²  : {base_r2:.4f} ± {base_r2_std:.4f}")

# OLS Summary
X_final = df_train[selected_features].values
X_with_const = sm.add_constant(X_final)
final_model = sm.OLS(y, X_with_const).fit()
print(final_model.summary())

[Initial] Features: ['Open', 'High', 'Low', 'Close', 'Volume', 'RSI_14', 'Momentum_10']
CV RMSE: 33.7506 ± 5.4155
CV R²  : 0.9375 ± 0.0458

Try dropping 'Open'
CV RMSE: 33.7118 ± 5.3096
→ Dropping 'Open' improved or maintained RMSE. Removing.

Try dropping 'High'
CV RMSE: 33.6441 ± 4.7762
→ Dropping 'High' improved or maintained RMSE. Removing.

Try dropping 'Low'
CV RMSE: 33.4557 ± 4.5349
→ Dropping 'Low' improved or maintained RMSE. Removing.

Try dropping 'Close'
CV RMSE: 459.0731 ± 274.5741
→ Dropping 'Close' worsened RMSE. Keeping.

Try dropping 'Volume'
CV RMSE: 31.9093 ± 2.2335
→ Dropping 'Volume' improved or maintained RMSE. Removing.

Try dropping 'RSI_14'
CV RMSE: 31.8912 ± 2.1734
→ Dropping 'RSI_14' improved or maintained RMSE. Removing.

Try dropping 'Momentum_10'
CV RMSE: 31.9597 ± 2.1227
→ Dropping 'Momentum_10' worsened RMSE. Keeping.

=== Final Selected Features ===
Features: ['Close', 'Momentum_10']
CV RMSE: 31.8912 ± 5.4155
CV R²  : 0.9399 ± 0.0458
                   

/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use 

In [37]:
# Define all candidates (original + engineered)
all_possible_features = ['Open', 'High', 'Low', 'Volume', 'RSI_14']

# Find features that were previously dropped
remaining_candidates = [f for f in all_possible_features if f not in selected_features]

print("\n=== Start Trying to Add Remaining Features ===")

for feature in remaining_candidates:
    trial_features = selected_features + [feature]
    X_trial = df_train[trial_features].values

    # Cross-validation performance
    trial_rmse, trial_std, trial_r2, trial_r2_std = cv_rmse_r2(X_trial, y)

    # Statistical significance check
    X_with_const = sm.add_constant(X_trial)
    sm_model = sm.OLS(y, X_with_const).fit()
    p_value = sm_model.pvalues[-1]  # Only check p-value of newly added feature

    print(f"\nTry adding '{feature}'")
    print(f"CV RMSE: {trial_rmse:.4f} ± {trial_std:.4f}, p-value: {p_value:.4f}")

    # Append result to table regardless of whether we keep the feature
    if trial_rmse < base_rmse and p_value < 0.05:
        print(f"→ Adding '{feature}' improves RMSE and is statistically significant. Appending.")
        selected_features.append(feature)
        base_rmse, base_std = trial_rmse, trial_std
        base_r2, base_r2_std = trial_r2, trial_r2_std
        step_label = f"Re-added {feature}"
    else:
        print(f"→ Adding '{feature}' does not help. Skipping.")
        step_label = f"Tried re-adding {feature} (not used)"

    # Record step in engineered_result_table
    engineered_result_table.loc[len(engineered_result_table)] = {
        'Step': step_label,
        'Features Used': ', '.join(selected_features),
        'CV RMSE': f"{trial_rmse:.4f} ± {trial_std:.4f}",
        'CV R²': f"{trial_r2:.4f} ± {trial_r2_std:.4f}"
    }

/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use 


=== Start Trying to Add Remaining Features ===

Try adding 'Open'
CV RMSE: 31.8737 ± 2.1760, p-value: 0.1041
→ Adding 'Open' does not help. Skipping.

Try adding 'High'
CV RMSE: 31.8556 ± 2.4612, p-value: 0.0004
→ Adding 'High' improves RMSE and is statistically significant. Appending.

Try adding 'Low'
CV RMSE: 32.3942 ± 3.3156, p-value: 0.1900
→ Adding 'Low' does not help. Skipping.

Try adding 'Volume'
CV RMSE: 33.1627 ± 4.5203, p-value: 0.3172
→ Adding 'Volume' does not help. Skipping.

Try adding 'RSI_14'
CV RMSE: 31.8952 ± 2.5473, p-value: 0.2847
→ Adding 'RSI_14' does not help. Skipping.


In [38]:
# Prepare data
X_train_final = df_train[selected_features].values
y_train_final = df_train['Target_Close'].values

X_test = df_test[selected_features].values
y_test = df_test['Target_Close'].values

# Train model
model = LinearRegression()
model.fit(X_train_final, y_train_final)

# Predict on test set
y_pred = model.predict(X_test)

# Evaluate
test_rmse = mean_squared_error(y_test, y_pred, squared=False)
test_mae = mean_absolute_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)
# Record final test performance in the result table
engineered_result_table.loc[len(engineered_result_table)] = {
    'Step': 'Final Test Evaluation',
    'Features Used': ', '.join(selected_features),
    'CV RMSE': 'N/A (Test Only)',
    'CV R²': f"{test_r2:.4f} (Test)"
}
print(f"Test RMSE : {test_rmse:.4f}")
print(f"Test MAE  : {test_mae:.4f}")
print(f"Test R²   : {test_r2:.4f}")

Test RMSE : 420.2378
Test MAE  : 400.8547
Test R²   : -26.5006


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [39]:
engineered_result_table

,Step,Features Used,CV RMSE,CV R²
0,All features (original + engineered),"Open, High, Low, Close, Volume, RSI_14, Moment...",33.7506 ± 5.4155,0.9375 ± 0.0458
1,After Feature Dropping,"Close, Momentum_10",31.8912 ± 5.4155,0.9399 ± 0.0458
2,Tried re-adding Open (not used),"Close, Momentum_10",31.8737 ± 2.1760,0.9401 ± 0.0463
3,Re-added High,"Close, Momentum_10, High",31.8556 ± 2.4612,0.9408 ± 0.0453
4,Tried re-adding Low (not used),"Close, Momentum_10, High",32.3942 ± 3.3156,0.9403 ± 0.0447
5,Tried re-adding Volume (not used),"Close, Momentum_10, High",33.1627 ± 4.5203,0.9384 ± 0.0462
6,Tried re-adding RSI_14 (not used),"Close, Momentum_10, High",31.8952 ± 2.5473,0.9408 ± 0.0451
7,Final Test Evaluation,"Close, Momentum_10, High",N/A (Test Only),-26.5006 (Test)


### poor estimation with rsi, momentum

In [41]:
# Create squared features
df_train['Open_sq'] = df_train['Open'] ** 2
df_train['High_sq'] = df_train['High'] ** 2
df_train['Low_sq'] = df_train['Low'] ** 2
df_train['Close_sq'] = df_train['Close'] ** 2
df_train['Volume_sq'] = df_train['Volume'] ** 2

# Same for test
df_test['Open_sq'] = df_test['Open'] ** 2
df_test['High_sq'] = df_test['High'] ** 2
df_test['Low_sq'] = df_test['Low'] ** 2
df_test['Close_sq'] = df_test['Close'] ** 2
df_test['Volume_sq'] = df_test['Volume'] ** 2

In [42]:
# Full feature set: original + engineered + squared terms
full_features = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'RSI_14', 'Momentum_10',
    'Open_sq', 'High_sq', 'Low_sq', 'Close_sq', 'Volume_sq'
]

X = df_train[full_features].values
y = df_train['Target_Close'].values

# Run CV
rmse, std, r2, r2_std = cv_rmse_r2(X, y)
poly_result_table = pd.DataFrame([{
    'Step': 'All features (including squared)',
    'Features Used': ', '.join(full_features),
    'CV RMSE': f"{rmse:.4f} ± {std:.4f}",
    'CV R²': f"{r2:.4f} ± {r2_std:.4f}"
}])
print("\n=== CV Performance with Squared Features ===")
print(f"Features: {full_features}")
print(f"CV RMSE: {rmse:.4f} ± {std:.4f}")
print(f"CV R²  : {r2:.4f} ± {r2_std:.4f}")


=== CV Performance with Squared Features ===
Features: ['Open', 'High', 'Low', 'Close', 'Volume', 'RSI_14', 'Momentum_10', 'Open_sq', 'High_sq', 'Low_sq', 'Close_sq', 'Volume_sq']
CV RMSE: 38.1808 ± 8.4771
CV R²  : 0.8924 ± 0.1293


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use 

In [43]:
# Start from full set including squared and engineered features
selected_features = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'RSI_14', 'Momentum_10',
    'Open_sq', 'High_sq', 'Low_sq', 'Close_sq', 'Volume_sq'
]

X = df_train[selected_features].values
y = df_train['Target_Close'].values

# Initial cross-validation performance
base_rmse, base_std, base_r2, base_r2_std = cv_rmse_r2(X, y)
print(f"[Initial] Features: {selected_features}")
print(f"CV RMSE: {base_rmse:.4f} ± {base_std:.4f}")
print(f"CV R²  : {base_r2:.4f} ± {base_r2_std:.4f}")

# Create result table for this experiment
poly_result_table = pd.DataFrame([{
    'Step': 'Initial (All)',
    'Features Used': ', '.join(selected_features),
    'CV RMSE': f"{base_rmse:.4f} ± {base_std:.4f}",
    'CV R²': f"{base_r2:.4f} ± {base_r2_std:.4f}"
}])

# Begin feature dropping with performance tracking
for feature in selected_features.copy():
    temp_features = [f for f in selected_features if f != feature]
    X_temp = df_train[temp_features].values
    temp_rmse, temp_std, temp_r2, temp_r2_std = cv_rmse_r2(X_temp, y)

    print(f"\nTry dropping '{feature}'")
    print(f"CV RMSE: {temp_rmse:.4f} ± {temp_std:.4f}")

    if temp_rmse <= base_rmse:
        print(f"→ Dropping '{feature}' improved or maintained RMSE. Removing.")
        selected_features = temp_features
        base_rmse, base_std = temp_rmse, temp_std
        base_r2, base_r2_std = temp_r2, temp_r2_std
        step_label = f"Dropped {feature}"
    else:
        print(f"→ Dropping '{feature}' worsened RMSE. Keeping.")
        step_label = f"Tried dropping {feature} (kept)"

    # Append result to poly_result_table
    poly_result_table.loc[len(poly_result_table)] = {
        'Step': step_label,
        'Features Used': ', '.join(selected_features),
        'CV RMSE': f"{temp_rmse:.4f} ± {temp_std:.4f}",
        'CV R²': f"{temp_r2:.4f} ± {temp_r2_std:.4f}"
    }

/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use 

[Initial] Features: ['Open', 'High', 'Low', 'Close', 'Volume', 'RSI_14', 'Momentum_10', 'Open_sq', 'High_sq', 'Low_sq', 'Close_sq', 'Volume_sq']
CV RMSE: 38.1808 ± 8.4771
CV R²  : 0.8924 ± 0.1293

Try dropping 'Open'
CV RMSE: 38.0017 ± 8.5902
→ Dropping 'Open' improved or maintained RMSE. Removing.

Try dropping 'High'
CV RMSE: 39.4874 ± 9.8380
→ Dropping 'High' worsened RMSE. Keeping.

Try dropping 'Low'
CV RMSE: 36.6065 ± 8.1715
→ Dropping 'Low' improved or maintained RMSE. Removing.

Try dropping 'Close'
CV RMSE: 36.8014 ± 8.3666
→ Dropping 'Close' worsened RMSE. Keeping.

Try dropping 'Volume'
CV RMSE: 36.2753 ± 8.0224
→ Dropping 'Volume' improved or maintained RMSE. Removing.

Try dropping 'RSI_14'
CV RMSE: 35.8240 ± 8.0217
→ Dropping 'RSI_14' improved or maintained RMSE. Removing.

Try dropping 'Momentum_10'
CV RMSE: 36.5331 ± 8.9531
→ Dropping 'Momentum_10' worsened RMSE. Keeping.


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use 


Try dropping 'Open_sq'
CV RMSE: 35.9811 ± 8.2964
→ Dropping 'Open_sq' worsened RMSE. Keeping.

Try dropping 'High_sq'
CV RMSE: 36.8849 ± 8.2112
→ Dropping 'High_sq' worsened RMSE. Keeping.

Try dropping 'Low_sq'
CV RMSE: 34.4341 ± 5.7091
→ Dropping 'Low_sq' improved or maintained RMSE. Removing.

Try dropping 'Close_sq'
CV RMSE: 34.7917 ± 5.6770
→ Dropping 'Close_sq' worsened RMSE. Keeping.

Try dropping 'Volume_sq'
CV RMSE: 34.6123 ± 5.9977
→ Dropping 'Volume_sq' worsened RMSE. Keeping.


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use 

In [44]:
print("\n=== Final Selected Features after Dropping ===")
print("Features:", selected_features)
print(f"CV RMSE: {base_rmse:.4f} ± {base_std:.4f}")
print(f"CV R²  : {base_r2:.4f} ± {base_r2_std:.4f}")

# OLS Summary
X_final = df_train[selected_features].values
X_with_const = sm.add_constant(X_final)
final_model = sm.OLS(y, X_with_const).fit()
print(final_model.summary())


=== Final Selected Features after Dropping ===
Features: ['High', 'Close', 'Momentum_10', 'Open_sq', 'High_sq', 'Close_sq', 'Volume_sq']
CV RMSE: 34.4341 ± 5.7091
CV R²  : 0.9150 ± 0.0954
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.996
Model:                            OLS   Adj. R-squared:                  0.995
Method:                 Least Squares   F-statistic:                 3.062e+04
Date:                Fri, 02 May 2025   Prob (F-statistic):               0.00
Time:                        03:34:42   Log-Likelihood:                -4639.9
No. Observations:                 971   AIC:                             9296.
Df Residuals:                     963   BIC:                             9335.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                 coef

In [45]:
# Prepare training and test data
X_train_final = df_train[selected_features].values
y_train_final = df_train['Target_Close'].values

X_test = df_test[selected_features].values
y_test = df_test['Target_Close'].values

# Train model on entire training set
model = LinearRegression()
model.fit(X_train_final, y_train_final)

# Predict on test set
y_pred = model.predict(X_test)

# Evaluate
test_rmse = mean_squared_error(y_test, y_pred, squared=False)
test_mae = mean_absolute_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)
poly_result_table.loc[len(poly_result_table)] = {
    'Step': 'Final Test Evaluation',
    'Features Used': ', '.join(selected_features),
    'CV RMSE': 'N/A (Test only)',
    'CV R²': f"{test_r2:.4f} (Test)"
}
print("\n=== Final Evaluation on Test Set ===")
print(f"Test RMSE : {test_rmse:.4f}")
print(f"Test MAE  : {test_mae:.4f}")
print(f"Test R²   : {test_r2:.4f}")


=== Final Evaluation on Test Set ===
Test RMSE : 419.6078
Test MAE  : 400.2211
Test R²   : -26.4182


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [46]:
poly_result_table

,Step,Features Used,CV RMSE,CV R²
0,Initial (All),"Open, High, Low, Close, Volume, RSI_14, Moment...",38.1808 ± 8.4771,0.8924 ± 0.1293
1,Dropped Open,"High, Low, Close, Volume, RSI_14, Momentum_10,...",38.0017 ± 8.5902,0.8933 ± 0.1295
2,Tried dropping High (kept),"High, Low, Close, Volume, RSI_14, Momentum_10,...",39.4874 ± 9.8380,0.8909 ± 0.1289
3,Dropped Low,"High, Close, Volume, RSI_14, Momentum_10, Open...",36.6065 ± 8.1715,0.8953 ± 0.1307
4,Tried dropping Close (kept),"High, Close, Volume, RSI_14, Momentum_10, Open...",36.8014 ± 8.3666,0.8933 ± 0.1343
5,Dropped Volume,"High, Close, RSI_14, Momentum_10, Open_sq, Hig...",36.2753 ± 8.0224,0.8971 ± 0.1284
6,Dropped RSI_14,"High, Close, Momentum_10, Open_sq, High_sq, Lo...",35.8240 ± 8.0217,0.8981 ± 0.1285
7,Tried dropping Momentum_10 (kept),"High, Close, Momentum_10, Open_sq, High_sq, Lo...",36.5331 ± 8.9531,0.8897 ± 0.1439
8,Tried dropping Open_sq (kept),"High, Close, Momentum_10, Open_sq, High_sq, Lo...",35.9811 ± 8.2964,0.8959 ± 0.1328
9,Tried dropping High_sq (kept),"High, Close, Momentum_10, Open_sq, High_sq, Lo...",36.8849 ± 8.2112,0.8976 ± 0.1258


In [47]:
# Prepare training and test data
X_train_final = df_train[['Close']].values
y_train_final = df_train['Target_Close'].values

X_test = df_test[['Close']].values
y_test = df_test['Target_Close'].values

# Train model on entire training set
model = LinearRegression()
model.fit(X_train_final, y_train_final)

# Predict on test set
y_pred = model.predict(X_test)

# Evaluate
test_rmse = mean_squared_error(y_test, y_pred, squared=False)
test_mae = mean_absolute_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)

print("\n=== Final Evaluation on Test Set ===")
print(f"Test RMSE : {test_rmse:.4f}")
print(f"Test MAE  : {test_mae:.4f}")
print(f"Test R²   : {test_r2:.4f}")


=== Final Evaluation on Test Set ===
Test RMSE : 420.8129
Test MAE  : 401.4054
Test R²   : -26.5759


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


### Simple is best it is hard to develop model with appending features...